# Gold Layer - Pipeline Metrics by Window

This notebook builds a Gold-layer summary table that combines transit and weather Silver metrics at the time-window level.

The goal is to provide a lightweight operational view of the telemetry pipeline, including:

- event volume per source
- ingestion latency signals
- basic source coverage across time windows

The output table is written to:

`gold_pipeline_metrics_window`

In [0]:
_ = spark.sql("USE azure_streaming_mvp")

## Build Gold Pipeline metrics (window level)

Aggregate transit and weather Silver metrics by time window and combine them into a single Gold table for operational monitoring.

In [0]:
%sql
CREATE OR REPLACE TABLE gold_pipeline_metrics_window
USING DELTA
AS

WITH transit AS (
  SELECT
    window_start,
    window_end,
    SUM(n_events) AS transit_total_events,
    AVG(avg_ingest_delay_sec) AS transit_avg_ingest_delay_sec
  FROM silver_transit_metrics
  GROUP BY window_start, window_end
),
weather AS (
  SELECT
    window_start,
    window_end,
    SUM(n_events) AS weather_total_events,
    AVG(avg_ingest_delay_sec) AS weather_avg_ingest_delay_sec
  FROM silver_weather_metrics
  GROUP BY window_start, window_end
)
SELECT
  COALESCE(t.window_start, w.window_start) AS window_start,
  COALESCE(t.window_end , w.window_end) AS window_end,
  t.transit_total_events,
  w.weather_total_events,
  t.transit_avg_ingest_delay_sec,
  w.weather_avg_ingest_delay_sec
FROM transit t
FULL OUTER JOIN weather w
ON t.window_start = w.window_start
AND t.window_end = w.window_end;

num_affected_rows,num_inserted_rows


## Preview pipeline metrics

In [0]:
%sql
SELECT *
FROM gold_pipeline_metrics_window
ORDER BY window_start DESC
LIMIT 20;

window_start,window_end,transit_total_events,weather_total_events,transit_avg_ingest_delay_sec,weather_avg_ingest_delay_sec
2026-03-11T12:05:00.000Z,2026-03-11T12:10:00.000Z,21,null,119.15277777777777,null
2026-03-11T12:00:00.000Z,2026-03-11T12:05:00.000Z,16,1,409.69696969696975,575.0
2026-03-11T11:55:00.000Z,2026-03-11T12:00:00.000Z,20,null,659.9777777777778,null
2026-03-11T11:50:00.000Z,2026-03-11T11:55:00.000Z,16,null,943.5,null
2026-03-11T11:45:00.000Z,2026-03-11T11:50:00.000Z,21,1,1275.9242424242423,1175.0
2026-03-11T11:40:00.000Z,2026-03-11T11:45:00.000Z,14,null,1553.3703703703702,null
2026-03-11T11:35:00.000Z,2026-03-11T11:40:00.000Z,18,null,1818.2727272727273,null
2026-03-11T11:30:00.000Z,2026-03-11T11:35:00.000Z,14,2,2124.0476190476193,2075.0
2026-03-11T11:25:00.000Z,2026-03-11T11:30:00.000Z,18,null,2460.275,null
2026-03-11T11:20:00.000Z,2026-03-11T11:25:00.000Z,12,null,2769.3,null


In [0]:
# ---- Notebook completion signal ----
dbutils.notebook.exit("OK")